# Convex Cost Structure vs RL Pricing Accuracy
This notebook explores how the convex cost parameters (`c` and `gamma`) relate to the percent difference between LSM and RL swing option prices.

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

sns.set_theme(style="whitegrid")

In [2]:
data_path = "Convex Costs Results 8.csv"
df = pd.read_csv(data_path)

benchmark_col = 'PctDiff_mean'
benchmark_label = 'PctDiff_mean = (RL_mean - LSM_full) / LSM_full'
lsm_col = 'LSM_full'

df.head()

,Configuration,c,gamma,LSM_full,LSM_full_CI95,RL_seed11,RL_seed12,RL_seed13,RL_seed14,RL_seed15,...,RL_seed25,RL_mean,RL_std,RL_best,RL_best_seed,PctDiff_mean,PctDiff_best,PctDiff_CI95,RL_BangBangness_mean,LSM_BangBangness
0,SwingOption_20_c0.00_gamma1,0.00,1.0,2.667581,0.021880,NaN,NaN,2.626509,NaN,NaN,...,NaN,2.626509,0.000000,2.626509,13,-1.539640,-1.539640,NaN,0.829482,0.999674
1,SwingOption_20_c0.01_gamma1,0.01,1.0,2.554873,0.021689,2.537183,2.531841,2.532728,NaN,NaN,...,NaN,2.533917,0.002863,2.537183,11,-0.820242,-0.692426,0.126791,0.684309,0.999642
2,SwingOption_20_c0.01_gamma1.5,0.01,1.5,2.513135,0.021365,2.489274,2.486036,2.496135,NaN,NaN,...,NaN,2.490482,0.005157,2.496135,13,-0.901401,-0.676448,0.232197,0.647668,0.738344
3,SwingOption_20_c0.01_gamma2,0.01,2.0,2.460907,0.021122,2.425148,2.435098,2.436869,NaN,NaN,...,NaN,2.432372,0.006318,2.436869,13,-1.159549,-0.976803,0.290532,0.504338,0.568229
4,SwingOption_20_c0.01_gamma3,0.01,3.0,2.328916,0.020411,2.290842,2.312991,2.292833,NaN,NaN,...,NaN,2.298889,0.012253,2.312991,12,-1.289307,-0.683786,0.595379,0.277695,0.256161


## Summary Statistics
Quick descriptive statistics for the main variables.

In [3]:
df.describe()

,c,gamma,LSM_full,LSM_full_CI95,RL_seed11,RL_seed12,RL_seed13,RL_seed14,RL_seed15,RL_seed16,...,RL_seed25,RL_mean,RL_std,RL_best,RL_best_seed,PctDiff_mean,PctDiff_best,PctDiff_CI95,RL_BangBangness_mean,LSM_BangBangness
count,26.000000,26.000000,26.000000,26.000000,25.000000,25.000000,26.000000,1.000000,1.000000,1.000000,...,1.000000,26.000000,26.000000,26.000000,26.000000,26.000000,26.000000,25.000000,26.000000,26.000000
mean,0.056538,1.711538,1.944968,0.018998,1.897593,1.899406,1.926063,1.956277,1.960128,1.966138,...,1.941712,1.926488,0.006227,1.931974,12.384615,-0.879957,-0.586306,0.380157,0.379098,0.495500
std,0.045602,0.680780,0.457849,0.002005,0.433228,0.435770,0.448257,NaN,NaN,NaN,...,NaN,0.448736,0.004391,0.449061,1.387859,0.754606,0.791378,0.244821,0.265871,0.386931
min,0.000000,1.000000,1.032112,0.014110,1.045639,1.052585,1.044349,1.956277,1.960128,1.966138,...,1.941712,1.047524,0.000000,1.052585,11.000000,-2.008960,-1.714728,0.027739,0.014445,0.020816
25%,0.020000,1.000000,1.668654,0.017783,1.643047,1.624800,1.652486,1.956277,1.960128,1.966138,...,1.941712,1.648914,0.003432,1.654606,12.000000,-1.279986,-0.992257,0.210688,0.153949,0.167352
50%,0.045000,1.500000,1.984536,0.019297,1.965019,1.960285,1.946043,1.956277,1.960128,1.966138,...,1.941712,1.961199,0.004918,1.973110,12.000000,-1.076560,-0.729055,0.301300,0.319461,0.401359
75%,0.080000,2.000000,2.317229,0.020495,2.250320,2.258397,2.282642,1.956277,1.960128,1.966138,...,1.941712,2.287565,0.008459,2.299342,13.000000,-0.788792,-0.427232,0.581861,0.638261,0.999666
max,0.150000,3.000000,2.667581,0.021880,2.537183,2.531841,2.626509,1.956277,1.960128,1.966138,...,1.941712,2.626509,0.017692,2.626509,18.000000,1.493309,1.983664,0.950299,0.829482,0.999956


## Pairwise Spearman Correlations
Spearman rank correlations capture monotonic relationships without assuming linearity.

In [4]:
spearman_corr = df[['c', 'gamma', benchmark_col]].corr(method='spearman')
spearman_corr

,c,gamma,PctDiff_mean
c,1.000000,-0.079429,0.225949
gamma,-0.079429,1.000000,0.153185
PctDiff_mean,0.225949,0.153185,1.000000


## Scatter Visualization
Visualizing how the default paper-facing RL vs LSM_full comparison varies with `c` and `gamma`.

In [5]:
from itertools import cycle
from plotly.subplots import make_subplots
from plotly.colors import qualitative
import plotly.graph_objects as go
import plotly.io as pio
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

pio.renderers.default = "notebook_connected"

fig_plotly = make_subplots(
    rows=2, cols=2,
    subplot_titles=[f'{benchmark_col} vs c', f'{benchmark_col} vs gamma', '', ''],
    shared_yaxes=True,
    vertical_spacing=0.10,
    row_heights=[0.84, 0.16]
)

palette = qualitative.Plotly + qualitative.Safe

for idx, x_col in enumerate(['c', 'gamma'], start=1):
    color_var = 'gamma' if x_col == 'c' else 'c'
    unique_values = sorted(df[color_var].unique())
    palette_iter = cycle(palette)
    color_map = {val: next(palette_iter) for val in unique_values}

    for val in unique_values:
        mask = df[color_var] == val
        fig_plotly.add_trace(
            go.Scatter(
                x=df.loc[mask, x_col],
                y=df.loc[mask, benchmark_col],
                mode='markers',
                marker=dict(size=8, color=color_map[val]),
                hoverinfo='skip',
                showlegend=False
            ),
            row=1, col=idx
        )

    x_range = np.linspace(df[x_col].min(), df[x_col].max(), 200)
    pred_frame = pd.DataFrame({x_col: x_range})
    trend_fit = smf.ols(f'{benchmark_col} ~ {x_col}', data=df).fit()
    pred_summary = trend_fit.get_prediction(pred_frame).summary_frame()
    mean = pred_summary['mean']
    se = pred_summary['mean_se']
    upper = mean + se
    lower = mean - se

    fig_plotly.add_trace(
        go.Scatter(
            x=np.concatenate([x_range, x_range[::-1]]),
            y=np.concatenate([upper, lower[::-1]]),
            fill='toself',
            fillcolor='rgba(31, 119, 180, 0.2)',
            line=dict(color='rgba(255,255,255,0)'),
            hoverinfo='skip',
            mode='lines',
            showlegend=False
        ),
        row=1, col=idx
    )

    fig_plotly.add_trace(
        go.Scatter(
            x=x_range, y=mean, mode='lines',
            line=dict(color='rgba(31, 119, 180, 1)', width=2),
            name='Trend', showlegend=False
        ),
        row=1, col=idx
    )

    fig_plotly.update_xaxes(title_text=x_col, row=1, col=idx)

    legend_x = list(range(len(unique_values)))
    legend_y = [0] * len(unique_values)
    legend_colors = [color_map[v] for v in unique_values]
    legend_labels = [format(v, '.3g') for v in unique_values]

    fig_plotly.add_trace(
        go.Scatter(
            x=legend_x, y=legend_y,
            mode='markers+text',
            marker=dict(color=legend_colors, size=12),
            text=legend_labels, textposition='bottom center',
            hoverinfo='skip', showlegend=False
        ),
        row=2, col=idx
    )

    fig_plotly.add_annotation(
        x=0.5, y=1.02, xref=f'x{idx+2} domain', yref='y domain',
        text=f"<b>{color_var}</b>", showarrow=False, font=dict(size=12),
        row=2, col=idx
    )

    fig_plotly.update_xaxes(visible=False, row=2, col=idx)
    fig_plotly.update_yaxes(visible=False, row=2, col=idx)

y_axis_title = rf"${benchmark_col} = \frac{{\mathrm{{RL}} - \mathrm{{LSM_{{full}}}}}}{{\mathrm{{LSM_{{full}}}}}}$"
fig_plotly.update_yaxes(title_text=y_axis_title, row=1, col=1, title_standoff=14, automargin=True)

fig_plotly.update_layout(
    title=dict(
        text='RL vs LSM_full Pricing Gap under Convex Costs',
        x=0.5, xanchor='center',
        y=0.98, yanchor='top',
        pad=dict(t=2, b=0)
    ),
    template='plotly_white',
    height=600, width=980,
    margin=dict(l=80, r=30, t=70, b=50),
    font=dict(size=12)
)

fig_plotly.update_xaxes(title_standoff=10, row=1, col=1)
fig_plotly.update_xaxes(title_standoff=10, row=1, col=2)
fig_plotly.update_yaxes(title_standoff=14, row=1, col=1)

fig_plotly.show(renderer="jupyterlab")

## Robust Linear Model
Fit an ordinary least squares model with heteroskedasticity-robust standard errors to test whether `c`, `gamma`, and their interaction explain the default RL vs LSM_full percentage gap.

In [6]:
formula = f'{benchmark_col} ~ c + gamma + c:gamma'
model = smf.ols(formula, data=df).fit()
robust_res = model.get_robustcov_results(cov_type='HC3')
robust_res.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:           PctDiff_mean   R-squared:                       0.476
Model:                            OLS   Adj. R-squared:                  0.404
Method:                 Least Squares   F-statistic:                     2.658
Date:                Wed, 08 Apr 2026   Prob (F-statistic):             0.0735
Time:                        20:29:35   Log-Likelihood:                -20.663
No. Observations:                  26   AIC:                             49.33
Df Residuals:                      22   BIC:                             54.36
Df Model:                           3                                         
Covariance Type:                  HC3                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -0.9268      0.438     -2.118      0.046      -1.834      -0.019
c            -19.2011     12.287     -1.563      0.132     -44.684       6.282
gamma         -0.2145      0.254     -0.845      0.407      -0.741       0.312
c:gamma       16.3477      8.028      2.036      0.054      -0.301      32.996
==============================================================================
Omnibus:                        5.852   Durbin-Watson:                   2.640
Prob(Omnibus):                  0.054   Jarque-Bera (JB):                3.881
Skew:                          -0.805   Prob(JB):                        0.144
Kurtosis:                       3.996   Cond. No.                         176.
==============================================================================

Notes:
[1] Standard Errors are heteroscedasticity robust (HC3)
"""

## Effect Grid
Predicted RL vs LSM_full percent difference across the observed grid of `c` and `gamma`.

In [7]:
grid = df[['c', 'gamma']].drop_duplicates().sort_values(['c', 'gamma']).reset_index(drop=True)
grid['pred_pct_diff'] = robust_res.predict(grid)
grid

,c,gamma,pred_pct_diff
0,0.00,1.0,-1.141320
1,0.01,1.0,-1.169855
2,0.01,1.5,-1.195363
3,0.01,2.0,-1.220871
4,0.01,3.0,-1.271887
5,0.02,1.0,-1.198389
6,0.02,1.5,-1.142159
7,0.02,2.0,-1.085928
8,0.02,3.0,-0.973468
9,0.04,1.0,-1.255458


## Interpretation
- The regression provides coefficient tests for `c`, `gamma`, and their interaction under robust standard errors.
- Combine the coefficient p-values with the Spearman correlations to assess whether higher convex costs correspond to larger RL advantages relative to `LSM_full`.
- Inspect the predicted grid to pinpoint regimes where RL diverges most from the tuned full-state LSM benchmark.

In [8]:
print(f"Robust Regression Coefficients ({benchmark_col} ~ c + gamma + c:gamma):\n")
for name, value in zip(robust_res.model.exog_names, robust_res.params):
    if name == "Intercept":
        desc = f"Baseline {benchmark_col} when c and gamma are zero"
    elif name == "c":
        desc = f"Effect of c (convex cost parameter) on {benchmark_col}"
    elif name == "gamma":
        desc = f"Effect of gamma (convexity exponent) on {benchmark_col}"
    elif name == "c:gamma":
        desc = "Interaction effect: how c's effect changes with gamma"
    else:
        desc = ""
    print(f"{name:10}: {value:10.4f}   # {desc}")

Robust Regression Coefficients (PctDiff_mean ~ c + gamma + c:gamma):

Intercept :    -0.9268   # Baseline PctDiff_mean when c and gamma are zero
c         :   -19.2011   # Effect of c (convex cost parameter) on PctDiff_mean
gamma     :    -0.2145   # Effect of gamma (convexity exponent) on PctDiff_mean
c:gamma   :    16.3477   # Interaction effect: how c's effect changes with gamma


In [9]:
robust_res.pvalues

array([0.04570291, 0.13240457, 0.4070392 , 0.05392452])

In [10]:
pctdiff_pivot = (
    df.pivot_table(index='c', columns='gamma', values=benchmark_col, aggfunc='mean')
      .sort_index()
      .reindex(sorted(df['gamma'].unique()), axis=1)
)

abs_max = np.nanmax(np.abs(pctdiff_pivot.values))

def border_color(val):
    if pd.isna(val):
        return ''
    color = 'green' if val > 0 else 'red'
    return f'border: 2px solid {color};'

styled_pivot = (
    pctdiff_pivot.round(3)
    .style.format(precision=3)
    .background_gradient(cmap='coolwarm', axis=None, vmin=-abs_max, vmax=abs_max)
    .map(border_color)
)
styled_pivot

gamma,1.000000,1.500000,2.000000,3.000000
c,,,,
0.000000,-1.540,nan,nan,nan
0.010000,-0.820,-0.901,-1.160,-1.289
0.020000,-1.189,-1.171,-1.252,-1.097
0.040000,-0.778,-1.614,-1.056,-0.079
0.050000,-0.973,-1.296,-1.327,0.558
0.080000,-1.001,-1.766,-0.128,nan
0.100000,-0.494,-0.995,0.131,nan
0.150000,-1.125,-2.009,1.493,nan
